In [1]:
import pandas as pd
import seaborn as sns
from sklearn.metrics import precision_score,accuracy_score,recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier


In [2]:
heart_df = pd.read_csv("heart.csv")
heart_df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [3]:
x = heart_df.drop("target",axis=1)
y = heart_df["target"]

x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size = 0.2,random_state = 42
)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

knn_classifier = KNeighborsClassifier(n_neighbors = 7)
knn_classifier.fit(x_train_scaled,y_train)

y_pred = knn_classifier.predict(x_test_scaled)
print("recall score : ",recall_score(y_test,y_pred))
print("accuracy score : ",accuracy_score(y_test,y_pred))
print("precision score : ",precision_score(y_test,y_pred))

recall score :  0.90625
accuracy score :  0.9180327868852459
precision score :  0.9354838709677419


In [9]:
# cross validation for finding best huperparameter....

from sklearn.model_selection import GridSearchCV

classifier = KNeighborsClassifier()
param_grid = {"n_neighbors" : [3,5,7,9,11,13,15,17,19,21,23,25]}
classifier_cv = GridSearchCV(
    classifier,
    param_grid,
    cv = 5,
    scoring = "recall"
)
classifier_cv.fit(x_train_scaled,y_train)
y2_pred = classifier_cv.predict(x_test_scaled)
print("recall score is : ",recall_score(y_test,y2_pred))
print("accuracy_score is : ",accuracy_score(y_test,y2_pred))
print("precision score is : ",precision_score(y_test,y2_pred))

res = pd.DataFrame(classifier_cv.cv_results_)
print(res[["param_n_neighbors","mean_test_score"]])

print(classifier_cv.best_params_)

recall score is :  0.90625
accuracy_score is :  0.8852459016393442
precision score is :  0.8787878787878788
    param_n_neighbors  mean_test_score
0                   3         0.864387
1                   5         0.857550
2                   7         0.871795
3                   9         0.856980
4                  11         0.871510
5                  13         0.894302
6                  15         0.909402
7                  17         0.909402
8                  19         0.924501
9                  21         0.932194
10                 23         0.932194
11                 25         0.924501
{'n_neighbors': 21}


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    "knn__n_neighbors": [3,5,7,9,11,13,15,17,19,21,23,25]
}

classifier_cv = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="recall_weighted"
)

classifier_cv.fit(x_train, y_train)

y3_pred = classifier_cv.predict(x_test)

print("Precision:", precision_score(y_test, y3_pred, average='weighted'))
print("Recall:", recall_score(y_test, y3_pred, average='weighted'))
print("Accuracy:", accuracy_score(y_test, y3_pred))

res = pd.DataFrame(classifier_cv.cv_results_)
print(res[["param_knn__n_neighbors", "mean_test_score"]])

print("Best Params:", classifier_cv.best_params_)

Precision: 0.8854765453126109
Recall: 0.8852459016393442
Accuracy: 0.8852459016393442
    param_knn__n_neighbors  mean_test_score
0                        3         0.797534
1                        5         0.814116
2                        7         0.809949
3                        9         0.801616
4                       11         0.797449
5                       13         0.822449
6                       15         0.826531
7                       17         0.818282
8                       19         0.818282
9                       21         0.830697
10                      23         0.818282
11                      25         0.814116
Best Params: {'knn__n_neighbors': 21}
